# D1｜Loss 與梯度（Google AI Application Engineer 14 天衝刺）

**驗收標準**：用 NumPy 寫出穩定版 softmax cross-entropy，回傳 `loss` 與 `dlogits`，不用 autograd；finite difference 誤差 < 1e-6。

**使用方式**
1. 先念書，再蓋起筆記做這份 notebook。
2. 只改有 `# TODO` 的格子；測試格不要改。
3. 由上往下執行，每個測試印出 `PASS` 才往下。
4. 最後填「當日紀錄」，再貼回 Notion 的錯題頁。

| 段落 | 建議時間 |
|---|---|
| 0. 口述題（蓋筆記） | 15 分 |
| 1. stable softmax | 20 分 |
| 2. cross-entropy loss + 梯度 | 40 分 |
| 3. 邊界測試 | 15 分 |
| 4. finite difference 驗證 | 30 分 |
| 5. 追問：mean vs sum | 15 分 |
| 6. 英文口述＋紀錄 | 30 分 |

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(0)

def check(name, cond):
    print(("PASS  " if cond else "FAIL  ") + name)
    return bool(cond)

## 0. 口述題（先寫在這格，蓋筆記作答）

1. logits、softmax 機率、one-hot 標籤 y 各是什麼？為什麼模型通常輸出 logits？
2. MSE、BCE、CE 各適合什麼任務？分類用 MSE 有什麼問題？
3. 寫出 softmax＋CE 對 logits 的梯度（batch 取 mean），形狀是什麼？
4. 為什麼 softmax 要先減掉每列最大值？會不會改變結果？

**我的回答：**

1. 
2. 
3. 
4. 

## 1. Stable softmax

- 輸入 `logits`：shape `(N, C)`，float
- 輸出 `probs`：shape `(N, C)`，每列總和為 1
- 不可出現 `inf` 或 `nan`，即使 logits 是 ±1000

In [ ]:
def stable_softmax(logits):
    # TODO: 每列減去最大值，再做 exp 與正規化
    raise NotImplementedError

In [ ]:
# 測試 1：與樸素版本比對（一般數值）、row sum、shift invariance、極端值
x = rng.normal(size=(4, 5))
naive = np.exp(x) / np.exp(x).sum(axis=1, keepdims=True)
p = stable_softmax(x)
check("shape 正確", p.shape == (4, 5))
check("與樸素版本一致", np.allclose(p, naive, atol=1e-12))
check("每列總和 = 1", np.allclose(p.sum(axis=1), 1.0))
check("每列加常數結果不變", np.allclose(stable_softmax(x + 123.0), p))
big = np.array([[1000.0, 0.0, -1000.0], [-1000.0, -1000.0, -1000.0]])
pb = stable_softmax(big)
check("±1000 仍有限", np.all(np.isfinite(pb)))
check("±1000 每列總和 = 1", np.allclose(pb.sum(axis=1), 1.0))
check("全相同 logits 為均勻分布", np.allclose(pb[1], 1/3))

## 2. Cross-entropy loss＋對 logits 的梯度

- 輸入 `logits`：`(N, C)`；`y`：`(N,)` 整數類別 0..C-1
- `reduction`：`"mean"`（預設）或 `"sum"`
- 輸出 `(loss, dlogits)`：`loss` 是純量，`dlogits` 形狀 `(N, C)`
- **loss 要用 log-sum-exp 直接算**，不要先算 softmax 再取 log（為什麼？寫在第 6 段）
- 不可用 autograd

In [ ]:
def softmax_cross_entropy(logits, y, reduction="mean"):
    # TODO:
    # 1) 每列減最大值得到 shifted
    # 2) log_probs = shifted - log(sum(exp(shifted)))
    # 3) loss = -log_probs[正確類別]，依 reduction 取 mean 或 sum
    # 4) dlogits = softmax - one_hot(y)，mean 時再除以 N
    raise NotImplementedError

In [ ]:
# 測試 2：手算小例子
logits = np.array([[2.0, 1.0, 0.1]])
y = np.array([0])
loss, g = softmax_cross_entropy(logits, y)
p = np.exp(logits) / np.exp(logits).sum()
check("loss 與手算一致", np.isclose(loss, -np.log(p[0, 0])))
check("梯度 = p - y", np.allclose(g, p - np.array([[1, 0, 0]])))
check("每列梯度總和 = 0", np.allclose(g.sum(axis=1), 0.0))

# 與樸素版本（一般數值）比對
X = rng.normal(size=(8, 6)); Y = rng.integers(0, 6, size=8)
P = np.exp(X) / np.exp(X).sum(axis=1, keepdims=True)
ref_loss = -np.log(P[np.arange(8), Y]).mean()
ref_g = (P - np.eye(6)[Y]) / 8
L, G = softmax_cross_entropy(X, Y)
check("batch loss 一致", np.isclose(L, ref_loss))
check("batch 梯度一致", np.allclose(G, ref_g))
check("梯度 shape = (N, C)", G.shape == X.shape)

## 3. 邊界測試

先自己想：還有哪些輸入會讓你的程式壞掉？在下方 TODO 格加上至少 1 個你自己的測試。

In [ ]:
# 極大 logits
L, G = softmax_cross_entropy(np.array([[1000.0, 0.0, -1000.0]]), np.array([2]))
check("極大 logits：loss 有限", np.isfinite(L))
check("極大 logits：loss ≈ 2000", np.isclose(L, 2000.0))
check("極大 logits：梯度有限", np.all(np.isfinite(G)))

# batch = 1
L1, G1 = softmax_cross_entropy(np.array([[0.5, -0.5]]), np.array([1]))
check("batch=1：梯度 shape (1, 2)", G1.shape == (1, 2))

# 每列加常數：loss 與梯度都不變
X = rng.normal(size=(5, 4)); Y = rng.integers(0, 4, size=5)
La, Ga = softmax_cross_entropy(X, Y)
Lb, Gb = softmax_cross_entropy(X + rng.normal(size=(5, 1)) * 50, Y)
check("每列加常數：loss 不變", np.isclose(La, Lb))
check("每列加常數：梯度不變", np.allclose(Ga, Gb))

# 完全預測正確
Lp, _ = softmax_cross_entropy(np.array([[100.0, 0.0]]), np.array([0]))
check("預測非常正確：loss ≈ 0 且 ≥ 0", 0 <= Lp < 1e-10)

In [ ]:
# TODO: 你自己的邊界測試（例如：C=1、y 越界時要不要報錯、float32 輸入）

## 4. Finite difference 驗證梯度

用中央差分：\( \frac{L(x+h e_i) - L(x-h e_i)}{2h} \)，`h = 1e-5`，float64。

驗收：最大相對誤差 < 1e-6。

In [ ]:
def numerical_grad(f, x, h=1e-5):
    # TODO: 對 x 的每個元素做中央差分，回傳與 x 同形狀的梯度
    raise NotImplementedError

def rel_error(a, b):
    return np.max(np.abs(a - b) / np.maximum(1e-8, np.abs(a) + np.abs(b)))

In [ ]:
for reduction in ["mean", "sum"]:
    X = rng.normal(size=(6, 5)); Y = rng.integers(0, 5, size=6)
    _, G = softmax_cross_entropy(X, Y, reduction=reduction)
    num = numerical_grad(lambda z: softmax_cross_entropy(z, Y, reduction=reduction)[0], X.copy())
    err = rel_error(G, num)
    print(f"{reduction}: 最大相對誤差 = {err:.2e}")
    check(f"finite difference ({reduction}) < 1e-6", err < 1e-6)

## 5. 追問：mean 換成 sum，梯度怎麼變？

先猜，再跑下面這格驗證。

In [ ]:
X = rng.normal(size=(32, 10)); Y = rng.integers(0, 10, size=32)
_, Gm = softmax_cross_entropy(X, Y, "mean")
_, Gs = softmax_cross_entropy(X, Y, "sum")
ratio = np.linalg.norm(Gs) / np.linalg.norm(Gm)
print("sum 梯度 / mean 梯度 =", ratio)
check("sum 梯度是 mean 的 N 倍", np.isclose(ratio, 32))

**我的解釋**（要講清楚適用條件）：

- sum 的梯度是 mean 的 ___ 倍。
- 對「其他條件相同的基本 SGD」，這相當於 ______。
- 換成 Adam、gradient clipping 或 weight decay 時，這個等價為什麼不一定成立？

## 6. 英文口述＋當日紀錄

**英文口述（錄音 90 秒）**：*Walk me through how you implement a numerically stable softmax cross-entropy and how you verify the gradient.*

**為什麼 loss 不先算 softmax 再取 log？**

**當日紀錄（貼回 Notion 錯題頁）**
- 日期／Day：2026-10-__／D1
- 題目：NumPy 穩定版 softmax cross-entropy＋梯度
- 練習模式：無提示／看一個提示／看完整解答
- 限時與實際耗時：
- 第一個錯誤決策：
- 最小失敗案例：
- 根因分類：
- 新增 regression test：
- 下次辨識訊號：
- 驗收結果：□ 全部 PASS　□ finite difference < 1e-6　□ 能口述 mean vs sum

## 延伸（選配）：兩層 MLP 反傳＋gradient check

`x → Linear(W1,b1) → ReLU → Linear(W2,b2) → softmax CE`，手寫反傳，對每個參數做 finite difference。

In [ ]:
# TODO（選配）